# Configuration

In [ ]:
%pip install tf-explain
%pip install tf-keras-vis
%pip install lime scikit-image
%pip install shap
%pip list

In [ ]:
# 1. SYSTEM & LOGGING (Suppress warnings, set seeds)
import os
import logging
import warnings
import gc
import glob
import random

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ['PYTHONHASHSEED'] = '42'
logging.getLogger('tensorflow').setLevel(logging.ERROR)
warnings.filterwarnings('ignore')

# 2. CORE DATA SCIENCE
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    confusion_matrix, classification_report, roc_auc_score,
    precision_recall_curve, auc, accuracy_score, f1_score,
    precision_score, recall_score
)
from imblearn.over_sampling import RandomOverSampler
from imblearn.under_sampling import RandomUnderSampler

# 3. DEEP LEARNING (TensorFlow/Keras)
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers, callbacks
from tensorflow.keras.applications import VGG16, ResNet50, MobileNetV2

# 4. EXPLAINABLE AI (XAI)
import shap
from lime import lime_image
from skimage.segmentation import mark_boundaries

# 5. REPRODUCIBILITY
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)
    os.environ['TF_DETERMINISTIC_OPS'] = '1'

set_seed(42)

# CONFIGURATION
RUN_MODE = "full"  # debug / full
DATASET_TYPE = "imbalanced"  # balanced / imbalanced
RUN_TRAINING = True
RUN_XAI = True
RUN_ANALYSIS = True

DATASET_PATH = "/kaggle/input/brain-tumor-mri-dataset"
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 16
LEARNING_RATE = 0.001
DROPOUT_RATE = 0.3
L2_REG = 0.001
RANDOM_SEED = 101

if RUN_MODE == "debug":
    N_FOLDS = 2
    MAX_EPOCHS = 3
    DEBUG_SAMPLES = 20
elif RUN_MODE == "screening":
    N_FOLDS = 3
    MAX_EPOCHS = 50
else:
    N_FOLDS = 5
    MAX_EPOCHS = 100

for d in ["results", "plots", "xai_outputs"]:
    os.makedirs(d, exist_ok=True)

In [ ]:

# Matplotlib defaults
plt.rcParams.update({"figure.dpi": 120, "savefig.bbox": "tight"})
# High resolution
plt.rcParams["figure.dpi"] = 300
plt.rcParams["savefig.dpi"] = 300

# Journal-size figures (inches)
plt.rcParams["figure.figsize"] = (6, 4)

# Clean professional fonts
plt.rcParams["font.size"] = 11
plt.rcParams["axes.titlesize"] = 12
plt.rcParams["axes.labelsize"] = 11
plt.rcParams["xtick.labelsize"] = 10
plt.rcParams["ytick.labelsize"] = 10
plt.rcParams["legend.fontsize"] = 10

# Remove top/right borders (clean look)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

# High-quality export
plt.rcParams["savefig.bbox"] = "tight"
plt.rcParams["savefig.format"] = "tiff"   # TIFF preferred for medical journals

# Seaborn clean style
sns.set_style("whitegrid")

print("✓ Global journal-quality plotting configured (300 DPI)")

## System Information¶

In [ ]:
import platform
import psutil

system_info = {
    "OS": platform.system(),
    "OS Version": platform.version(),
    "Python Version": platform.python_version(),
    "TensorFlow Version": tf.__version__,
    "NumPy Version": np.__version__,
    "CPU Count": os.cpu_count(),
    "Memory (GB)": psutil.virtual_memory().total / (1024**3),
    "GPU": len(tf.config.list_physical_devices('GPU')) > 0
}

with open("system_info.txt", "w") as f:
    for key, value in system_info.items():
        f.write(f"{key}: {value}\n")

print(system_info)

# Reproducibility Setup

In [ ]:

np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)
random.seed(RANDOM_SEED)
os.environ['PYTHONHASHSEED'] = str(RANDOM_SEED)
os.environ['TF_DETERMINISTIC_OPS'] = '1'

# Data Loading

In [ ]:
train_dir = os.path.join(DATASET_PATH, "Training")
test_dir = os.path.join(DATASET_PATH, "Testing")

class_names = sorted(os.listdir(train_dir))
num_classes = len(class_names)

train_paths = []
train_labels = []
for i, class_name in enumerate(class_names):
    class_path = os.path.join(train_dir, class_name)
    paths = glob.glob(os.path.join(class_path, "*.jpg"))
    train_paths.extend(paths)
    train_labels.extend([i] * len(paths))

test_paths = []
test_labels = []
for i, class_name in enumerate(class_names):
    class_path = os.path.join(test_dir, class_name)
    paths = glob.glob(os.path.join(class_path, "*.jpg"))
    test_paths.extend(paths)
    test_labels.extend([i] * len(paths))

if RUN_MODE == "debug":
    indices = np.random.choice(len(train_paths), min(DEBUG_SAMPLES, len(train_paths)), replace=False)
    train_paths = [train_paths[i] for i in indices]
    train_labels = [train_labels[i] for i in indices]

if DATASET_TYPE == "balanced":
    sampler = RandomOverSampler(random_state=RANDOM_SEED)
    train_paths_resampled, train_labels_resampled = sampler.fit_resample(np.array(train_paths).reshape(-1, 1), train_labels)
    train_paths = train_paths_resampled.flatten().tolist()
    train_labels = train_labels_resampled.tolist()

train_paths = np.array(train_paths)
train_labels = np.array(train_labels)
test_paths = np.array(test_paths)
test_labels = np.array(test_labels)

# Fold Creation

In [ ]:
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_SEED)
folds = list(skf.split(train_paths, train_labels))

# tf.data Pipeline

In [ ]:
def preprocess_image(image_path, label):
    image = tf.io.read_file(image_path)
    image = tf.image.decode_jpeg(image, channels=1)
    image = tf.image.convert_image_dtype(image, tf.float32)
    image = tf.image.resize(image, IMAGE_SIZE)
    image = tf.squeeze(image, axis=-1)
    image = tf.numpy_function(lambda x: cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8)).apply((x * 255).astype(np.uint8)), [image], tf.uint8)
    image = tf.numpy_function(lambda x: cv2.bilateralFilter(x, d=2, sigmaColor=50, sigmaSpace=50), [image], tf.uint8)
    image = tf.numpy_function(lambda x: cv2.applyColorMap(x, cv2.COLORMAP_BONE), [image], tf.uint8)
    # Convert to float32 and scale to [0,1] after all OpenCV ops
    image = tf.cast(image, tf.float32) / 255.0
    image = tf.image.per_image_standardization(image)
    # Ensure 3 channels for all models
    image = tf.image.grayscale_to_rgb(image) if tf.shape(image)[-1] == 1 else image
    # Explicitly set static shape for TensorFlow graph
    image.set_shape(IMAGE_SIZE + (3,))
    return image, label

def create_dataset(paths, labels, batch_size=BATCH_SIZE, shuffle=True):
    dataset = tf.data.Dataset.from_tensor_slices((paths, labels))
    if shuffle:
        dataset = dataset.shuffle(buffer_size=len(paths))
    dataset = dataset.map(preprocess_image, num_parallel_calls=tf.data.AUTOTUNE)
    dataset = dataset.batch(batch_size)
    dataset = dataset.prefetch(tf.data.AUTOTUNE)
    return dataset

# Model Definitions

In [ ]:
def create_custom_cnn(input_shape=IMAGE_SIZE + (3,), num_classes=num_classes):
    model = models.Sequential()
    filters = [32, 64, 128, 256, 512]
    for f in filters:
        model.add(layers.Conv2D(f, (3,3), padding='same', kernel_regularizer=tf.keras.regularizers.l2(L2_REG)))
        model.add(layers.BatchNormalization())
        model.add(layers.Activation('swish'))
        model.add(layers.MaxPooling2D((2,2)))
    model.add(layers.Flatten())
    model.add(layers.Dense(256, activation='relu', kernel_regularizer=tf.keras.regularizers.l2(L2_REG)))
    model.add(layers.Dropout(DROPOUT_RATE))
    model.add(layers.Dense(128, activation='relu', kernel_regularizer=tf.keras.regularizers.l2(L2_REG)))
    model.add(layers.Dropout(DROPOUT_RATE))
    model.add(layers.Dense(num_classes, activation='softmax'))
    return model

def create_vgg16(input_shape=IMAGE_SIZE + (3,), num_classes=num_classes):
    base = VGG16(weights='imagenet', include_top=False, input_shape=input_shape)
    base.trainable = False
    model = models.Sequential([
        base,
        layers.Flatten(),
        layers.Dense(64, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(num_classes, activation='softmax')
    ])
    return model

def create_resnet50(input_shape=IMAGE_SIZE + (3,), num_classes=num_classes):
    base = ResNet50(weights='imagenet', include_top=False, input_shape=input_shape)
    base.trainable = False
    model = models.Sequential([
        base,
        layers.GlobalAveragePooling2D(),
        layers.Dense(num_classes, activation='softmax')
    ])
    return model

def create_mobilenetv2(input_shape=IMAGE_SIZE + (3,), num_classes=num_classes):
    base = MobileNetV2(weights='imagenet', include_top=False, input_shape=input_shape)
    base.trainable = False
    model = models.Sequential([
        base,
        layers.GlobalAveragePooling2D(),
        layers.Dense(num_classes, activation='softmax')
    ])
    return model

if RUN_MODE == "debug":
    models_dict = {
        'custom_cnn': create_custom_cnn
    }
else:
    models_dict = {
        'custom_cnn': create_custom_cnn,
        'vgg16': create_vgg16,
        'resnet50': create_resnet50,
        'mobilenetv2': create_mobilenetv2
    }

# Training Engine

In [ ]:
def compile_model(model):
    model.compile(
        optimizer=optimizers.Adam(learning_rate=LEARNING_RATE),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

def train_model(model, train_ds, val_ds, epochs=MAX_EPOCHS):
    callbacks_list = [
        # Faster early stopping but still restore best weights
        callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True, verbose=1),
        # Reduce LR more aggressively to converge faster on plateaus
        callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-6, verbose=1),
        # Keep a checkpoint of best model to resume or inspect later
        callbacks.ModelCheckpoint(filepath=os.path.join('results', f'{model.name}_best.h5'), monitor='val_loss', save_best_only=True, save_weights_only=False, verbose=1),
        # Stop training if NaNs appear
        callbacks.TerminateOnNaN(),
        # Lightweight logging to CSV for quick inspection
        callbacks.CSVLogger(os.path.join('results', f'{model.name}_training_log.csv'))
    ]
    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=epochs,
        callbacks=callbacks_list,
        verbose=1
    )
    return model, history

# Cross-Validation Controller

In [ ]:
# -------------------------
# Metrics (fixes UndefinedMetricWarning)
# -------------------------
def compute_metrics(y_true, y_pred_classes, y_pred_probs):
    metrics = {}
    metrics['accuracy'] = accuracy_score(y_true, y_pred_classes)
    metrics['f1'] = f1_score(y_true, y_pred_classes, average='weighted', zero_division=0)
    metrics['precision'] = precision_score(y_true, y_pred_classes, average='weighted', zero_division=0)
    metrics['recall'] = recall_score(y_true, y_pred_classes, average='weighted', zero_division=0)
    metrics['confusion_matrix'] = confusion_matrix(y_true, y_pred_classes)

    # ROC-AUC (multiclass)
    try:
        if y_pred_probs.shape[1] > 1:
            metrics['roc_auc'] = roc_auc_score(y_true, y_pred_probs, multi_class='ovr')
        else:
            metrics['roc_auc'] = roc_auc_score(y_true, y_pred_probs[:, 0])
    except Exception:
        metrics['roc_auc'] = None

    # PR-AUC (multiclass)
    try:
        if y_pred_probs.shape[1] > 1:
            pr_auc = []
            for i in range(y_pred_probs.shape[1]):
                p, r, _ = precision_recall_curve((y_true == i).astype(int), y_pred_probs[:, i])
                pr_auc.append(auc(r, p))
            metrics['pr_auc'] = float(np.mean(pr_auc))
        else:
            p, r, _ = precision_recall_curve(y_true, y_pred_probs[:, 0])
            metrics['pr_auc'] = auc(r, p)
    except Exception:
        metrics['pr_auc'] = None

    return metrics

In [ ]:
import matplotlib.pyplot as plt
import cv2

def plot_gradcam_triplet(base_img, heatmap, overlay_img, sample_idx, output_path):
    plt.figure(figsize=(12, 4))
    plt.subplot(1, 3, 1)
    plt.imshow(base_img)
    plt.axis('off')
    plt.title('Base Image')

    plt.subplot(1, 3, 2)
    plt.imshow(heatmap, cmap='jet')
    plt.axis('off')
    plt.title('Grad-CAM Heatmap')

    plt.subplot(1, 3, 3)
    plt.imshow(overlay_img)
    plt.axis('off')
    plt.title('Overlay')

    plt.suptitle(f'Sample {sample_idx}')
    plt.tight_layout()
    plt.savefig(output_path)
    plt.show()


def plot_lime_triplet(base_img, lime_mask, overlay_img, sample_idx, output_path):
    plt.figure(figsize=(12, 4))
    plt.subplot(1, 3, 1)
    plt.imshow(base_img)
    plt.axis('off')
    plt.title('Base Image')

    plt.subplot(1, 3, 2)
    plt.imshow(lime_mask, cmap='jet')
    plt.axis('off')
    plt.title('LIME Mask')

    plt.subplot(1, 3, 3)
    plt.imshow(overlay_img)
    plt.axis('off')
    plt.title('Overlay')

    plt.suptitle(f'Sample {sample_idx}')
    plt.tight_layout()
    plt.savefig(output_path)
    plt.show()

# Faithfulness Metrics & XAI Robustness
This section implements quantitative faithfulness metrics (comprehensiveness, sufficiency, localization), compares XAI explanations on perturbed images, and aggregates XAI outputs for statistical analysis across models and folds.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from lime import lime_image
from skimage.segmentation import mark_boundaries
from tf_keras_vis.utils.model_modifiers import ReplaceToLinear
from tf_keras_vis.utils.scores import CategoricalScore
from tf_keras_vis.gradcam import Gradcam
from tf_keras_vis.gradcam_plus_plus import GradcamPlusPlus
from tf_keras_vis.scorecam import Scorecam

from tensorflow.keras import Model, Input

def sequential_to_functional(sequential_model):
    # Create a new Input layer with the same shape as the original model
    input_layer = Input(shape=sequential_model.input_shape[1:])
    # Pass the input through the Sequential model
    output_layer = sequential_model(input_layer)
    # Create a new Functional model
    return Model(inputs=input_layer, outputs=output_layer)

def get_last_conv_layer(model):
    # Recursively finds the last valid convolutional layer
    CONV_LIKE = (tf.keras.layers.Conv2D, tf.keras.layers.DepthwiseConv2D,
                 tf.keras.layers.SeparableConv2D, tf.keras.layers.Conv2DTranspose)
    last = None
    def walk(m):
        nonlocal last
        # If m is a Model or Sequential, iterate layers
        if hasattr(m, 'layers'):
            for layer in m.layers:
                if hasattr(layer, 'layers'):
                    walk(layer)
                if isinstance(layer, CONV_LIKE):
                    last = layer
    if isinstance(model, CONV_LIKE):
        return model
    walk(model)
    if last is None:
        raise ValueError("No valid Conv layer found.")
    return last

def make_gradcam_heatmap(model, img_batch, conv_layer, class_index=None):
    _ = model(img_batch, training=False)
    grad_model = tf.keras.Model(
        inputs=[model.input],
        outputs=[conv_layer.output, model.output]
    )
    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_batch, training=False)
        if class_index is None:
            class_index = tf.argmax(predictions[0])
        loss = predictions[:, class_index]
    grads = tape.gradient(loss, conv_outputs)[0]
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1))
    conv_outputs = conv_outputs[0]
    heatmap = conv_outputs @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0) / (tf.reduce_max(heatmap) + 1e-8)
    return heatmap.numpy(), int(class_index)

def save_gradcam_overlay(image, heatmap, path, alpha=0.4):
    import matplotlib.cm as cm
    heatmap = np.uint8(255 * heatmap)
    jet = cm.get_cmap("jet")
    jet_colors = jet(np.arange(256))[:, :3]
    jet_heatmap = jet_colors[heatmap]
    jet_heatmap = tf.keras.preprocessing.image.array_to_img(jet_heatmap)
    jet_heatmap = jet_heatmap.resize((image.shape[1], image.shape[0]))
    jet_heatmap = tf.keras.preprocessing.image.img_to_array(jet_heatmap)
    superimposed_img = jet_heatmap * alpha + image * 255
    superimposed_img = tf.keras.preprocessing.image.array_to_img(superimposed_img)
    superimposed_img.save(path)

def comprehensiveness_score(model, image, label, mask, baseline=0.0):
    """
    Comprehensiveness: Drop the most important region (mask=1) and see how much prediction confidence drops.
    """
    image_masked = image.copy()
    image_masked[mask == 1] = baseline
    pred_full = model.predict(np.expand_dims(image, 0))[0][label]
    pred_masked = model.predict(np.expand_dims(image_masked, 0))[0][label]
    return pred_full - pred_masked

def sufficiency_score(model, image, label, mask, baseline=0.0):
    """
    Sufficiency: Keep only the most important region (mask=1), set rest to baseline, see how much prediction remains.
    """
    image_masked = np.ones_like(image) * baseline
    image_masked[mask == 1] = image[mask == 1]
    pred_full = model.predict(np.expand_dims(image, 0))[0][label]
    pred_masked = model.predict(np.expand_dims(image_masked, 0))[0][label]
    return pred_masked / (pred_full + 1e-8)

xai_images, xai_labels, gradcam_heatmaps = [], [], []   # Global variables to hold XAI samples

def run_xai_suite(model, train_ds, test_ds, output_dir, model_name):
    print(f"[{model_name}] Initializing XAI Suite...")
    os.makedirs(f"{output_dir}/{model_name}", exist_ok=True)
    base_path = f"{output_dir}/{model_name}"
    TAKE_SAMPLE_EIMAGE_COUNT = 12
    if RUN_MODE == "debug":
        TAKE_SAMPLE_EIMAGE_COUNT = 2
    # Prepare samples for XAI
    sample_ds = test_ds.unbatch().take(TAKE_SAMPLE_EIMAGE_COUNT)
    xai_images, xai_labels = [], []
    for img, lbl in sample_ds:
        xai_images.append(img.numpy())
        xai_labels.append(np.argmax(lbl.numpy()))
    xai_images = np.array(xai_images)
    xai_labels = np.array(xai_labels)

    # --- Use inner model for Grad-CAM and SHAP ---
    # For custom_cnn, the inner model is model.layers[0]
    base_model = model.layers[0]
    print("[DEBUG] Using inner model:", base_model.name)
    img_batch = np.expand_dims(xai_images[0], axis=0)
    _ = base_model(img_batch, training=False)
    print("[DEBUG] Inner model called with real image batch to set input/output.")

    # Get last conv layer from inner model
    conv_layer = get_last_conv_layer(base_model)
    print(f"[{model_name}] Target Conv Layer: {conv_layer.name}")

    # GRAD-CAM
    print(f"[{model_name}] Running Grad-CAM...")
    for i in range(len(xai_images)):
        try:
            img_for_gradcam = np.expand_dims(xai_images[i], axis=0)
            heatmap, used_class = make_gradcam_heatmap(base_model, img_for_gradcam, conv_layer,
            xai_labels[i])

            gradcam_heatmaps.append(heatmap)  # Store for later robustness analysis

            overlay_path = f"{base_path}/gradcam_{i}.png"
            save_gradcam_overlay(xai_images[i], heatmap, overlay_path)
            print(f"[DEBUG] Overlay saved to: {overlay_path}")

            base_img = xai_images[i]
            overlay_img = plt.imread(overlay_path)
            # plot_gradcam_triplet is defined earlier and reused here
            plot_gradcam_triplet(
                base_img=base_img, heatmap=heatmap, overlay_img=overlay_img, sample_idx=i, output_path=f"{base_path}/gradcam_triplet_{i}.png" )
        except Exception as e:
            print(f"  [!] Grad-CAM failed for sample {i}: {e}")

    # LIME
    print(f"[{model_name}] Generating LIME...")
    try:
        explainer_lime = lime_image.LimeImageExplainer()
        for i in range(len(xai_images)):
            print(f"[DEBUG] Running LIME for sample {i}...")
            explanation = explainer_lime.explain_instance(
                xai_images[i].astype(np.float64),
                lambda x: model.predict(x, verbose=0),
                top_labels=1, hide_color=0, num_samples=500
            )
            temp, mask = explanation.get_image_and_mask(
                explanation.top_labels[0], positive_only=True, num_features=5, hide_rest=False
            )
            temp_vis = (temp - temp.min()) / (temp.max() - temp.min() + 1e-8)
            plt.figure(); plt.imshow(mark_boundaries(temp_vis, mask)); plt.axis("off")
            plt.savefig(f"{base_path}/lime_{i}.png", bbox_inches='tight'); plt.close()
            print(f"[DEBUG] LIME plot saved for sample {i}.")

            # --- LIME triplet visualization ---
            base_img = xai_images[i]
            lime_mask = mask.astype(float)
            lime_mask_norm = (lime_mask - lime_mask.min()) / (lime_mask.max() - lime_mask.min() + 1e-8)
            overlay_img = 0.6 * base_img + 0.4 * plt.cm.jet(lime_mask_norm)[..., :3]
            # plot_lime_triplet is defined earlier and reused here
            plot_lime_triplet(
                base_img=base_img,
                lime_mask=lime_mask,
                overlay_img=overlay_img,
                sample_idx=i,
                output_path=f"{base_path}/lime_triplet_{i}.png"
            )
    except Exception as e:
        print(f"  [!] LIME failed: {e}")


    # Example usage for a batch of images and masks (GradCAM/LIME/SHAP):
    faithfulness_results = []

    for i in range(len(xai_images)):
        img = xai_images[i]
        label = xai_labels[i]
        # Use GradCAM mask as example (thresholded heatmap)
        gradcam_heatmap = gradcam_heatmaps[i] # obtain from previous GradCAM output
        mask = (gradcam_heatmap > 0.5).astype(int)
        comp = comprehensiveness_score(model, img, label, mask)
        suff = sufficiency_score(model, img, label, mask)
        faithfulness_results.append({'sample': i, 'comprehensiveness': comp, 'sufficiency': suff})

    faithfulness_df = pd.DataFrame(faithfulness_results)
    faithfulness_df.to_csv(f"{base_path}/faithfulness_metrics.csv", index=False)
    faithfulness_df.head()

    # Compare explanations on perturbed images (robustness)
    robustness_results = []
    def perturb_image(image, mode='noise'):
        if mode == 'noise':
            noise = np.random.normal(0, 0.1, image.shape)
            return np.clip(image + noise, 0, 1)
        elif mode == 'rotate':
            return np.rot90(image)
        else:
            return image

    for i in range(len(xai_images)):
        img = xai_images[i]
        label = xai_labels[i]
        perturbed_img = perturb_image(img, mode='noise')
        # Get GradCAM for original and perturbed
        gradcam_heatmap_orig = gradcam_heatmaps[i]  # obtain from previous GradCAM output
        img_for_gradcam_pert = np.expand_dims(perturbed_img, axis=0)
        heatmap_pert, _ = make_gradcam_heatmap(base_model, img_for_gradcam_pert, conv_layer, label)
        gradcam_heatmap_pert = heatmap_pert
        # Similarity: e.g., Pearson correlation
        if gradcam_heatmap_orig.shape != gradcam_heatmap_pert.shape:
            gradcam_heatmap_pert = cv2.resize(gradcam_heatmap_pert, gradcam_heatmap_orig.shape[::-1])
            sim = np.corrcoef(gradcam_heatmap_orig.flatten(), gradcam_heatmap_pert.flatten())[0,1]
            robustness_results.append({'sample': i, 'similarity': sim})

    robustness_df = pd.DataFrame(robustness_results)
    robustness_df.to_csv(f"{base_path}/xai_robustness_metrics.csv", index=False)
    robustness_df.head()

# -------------------------
# UPDATED MAIN EXPERIMENT
# -------------------------
results = {}

def run_experiment(dataset_type):
    print(f"\n===== STARTING DATASET: {dataset_type} =====")

    output_dir = f"results/{dataset_type}"
    os.makedirs(output_dir, exist_ok=True)

    for model_name, model_func in models_dict.items():
        print(f"\n--- Training Model: {model_name} ---")
        fold_results = []

        # Last model trained will be used for final XAI
        last_trained_model = None

        for fold, (train_idx, val_idx) in enumerate(folds):
            # Create Fold Datasets
            train_ds = create_dataset(train_paths[train_idx], tf.keras.utils.to_categorical(train_labels[train_idx], num_classes))
            val_ds = create_dataset(train_paths[val_idx], tf.keras.utils.to_categorical(train_labels[val_idx], num_classes), shuffle=False)

            # Train
            model = model_func()
            model = compile_model(model)
            model, _ = train_model(model, train_ds, val_ds)

            # Evaluate Fold
            y_pred = model.predict(val_ds)
            metrics = compute_metrics(train_labels[val_idx], np.argmax(y_pred, axis=1), y_pred)
            metrics['fold'] = fold + 1
            fold_results.append(metrics)

            os.makedirs(f"{output_dir}/{model_name}", exist_ok=True)

            # Save metrics
            pd.DataFrame([metrics]).to_csv(f"{output_dir}/{model_name}/fold_{fold+1}_metrics.csv", index=False)

            last_trained_model = model # Store for XAI

        # Save Averaged Results
        results[model_name] = fold_results
        mean_metrics = {k: np.mean([r[k] for r in fold_results if isinstance(r[k], (int, float))])
                        for k in fold_results[0] if k not in ['confusion_matrix', 'fold']}
        pd.DataFrame([mean_metrics]).to_csv(f"{output_dir}/{model_name}/mean_metrics.csv", index=False)

        # FINAL STEP: Run XAI Suite on the last model instance
        test_ds = create_dataset(test_paths, tf.keras.utils.to_categorical(test_labels, num_classes), shuffle=False)

        # Use a fresh training DS for SHAP background images
        full_train_ds = create_dataset(train_paths, tf.keras.utils.to_categorical(train_labels, num_classes))

        run_xai_suite(last_trained_model, full_train_ds, test_ds, output_dir, model_name)

# Main experiment controller
if RUN_MODE == "full":
    for dataset_type in ["imbalanced", "balanced"]:
        run_experiment(dataset_type)
else:
    run_experiment(DATASET_TYPE)

# Final Retraining

In [ ]:
# ===========================
# FINAL TRAIN + TEST + ANALYSIS + XAI + STATS + SAVE-ALL-CSVS (ALL-IN-ONE)
# ===========================
# Assumes these already exist in your notebook:
# - results (dict): {model_name: [fold_metrics_dicts...]}
# - models_dict (dict): {model_name: callable -> builds model}
# - create_dataset(paths, onehot_labels, shuffle=True/False)
# - preprocess_image(path, label) -> (img_tensor, label_tensor) or similar
# - compile_model(model)
# - train_model(model, train_ds, val_ds_or_None, epochs) -> (model, history_or_logs)
# - compute_metrics(y_true, y_pred_classes, y_pred_probs) -> dict (must include 'accuracy' at least)
# - get_last_conv_layer(model) -> last conv layer object
# - train_paths, train_labels, test_paths, test_labels
# - num_classes, MAX_EPOCHS, RUN_ANALYSIS

import os
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt

import scipy.stats as stats
from scipy.stats import ttest_rel
from sklearn.metrics import confusion_matrix
from sklearn.calibration import calibration_curve

# ---------------------------
# 0) Folders
# ---------------------------
os.makedirs("results", exist_ok=True)
os.makedirs("plots", exist_ok=True)
os.makedirs("xai_outputs", exist_ok=True)

# ---------------------------
# 1) Pick best model by mean CV accuracy
# ---------------------------
best_model_name = max(
    results,
    key=lambda x: float(np.mean([r["accuracy"] for r in results[x] if "accuracy" in r]))
)
best_model_func = models_dict[best_model_name]
print("Best model selected:", best_model_name)

# ---------------------------
# 2) Build full train/test datasets
# ---------------------------
y_train_oh = tf.keras.utils.to_categorical(train_labels, num_classes)
y_test_oh  = tf.keras.utils.to_categorical(test_labels, num_classes)

train_ds_full = create_dataset(train_paths, y_train_oh)
test_ds = create_dataset(test_paths, y_test_oh, shuffle=False)

# ---------------------------
# 3) Final train on full train set (no val), save model, predict test
# ---------------------------
model = best_model_func()
model = compile_model(model)

model_dir = f"results/{best_model_name}"
os.makedirs(model_dir, exist_ok=True)

model, _ = train_model(model, train_ds_full, None, epochs=MAX_EPOCHS)  # no val for final fit
model.save(f"{model_dir}/final_best_model.h5")

y_pred = model.predict(test_ds)
y_pred_classes = np.argmax(y_pred, axis=1)
y_true = np.array(test_labels)

test_metrics = compute_metrics(y_true, y_pred_classes, y_pred)
pd.DataFrame([test_metrics]).to_csv(f"{model_dir}/test_results.csv", index=False)
print("Test metrics:", test_metrics)

# ==========================================================
# 4) SAVE ALL TEST-RELATED RESULTS (CSV + NPY where required)
# ==========================================================

# 4.1 Per-sample predictions (+ probabilities)
pred_df = pd.DataFrame({
    "image_path": test_paths,
    "true_label": y_true,
    "predicted_label": y_pred_classes
})
for i in range(y_pred.shape[1]):
    pred_df[f"prob_class_{i}"] = y_pred[:, i]
pred_df.to_csv(f"{model_dir}/per_sample_predictions.csv", index=False)

# 4.2 Confusion matrix
cm = confusion_matrix(y_true, y_pred_classes)
pd.DataFrame(cm).to_csv(f"{model_dir}/confusion_matrix.csv", index=False)

# 4.3 Calibration data (per class bins)
calibration_records = []
for class_idx in range(y_pred.shape[1]):
    y_true_bin = (y_true == class_idx).astype(int)
    y_prob = y_pred[:, class_idx]
    prob_true, prob_pred = calibration_curve(y_true_bin, y_prob, n_bins=10)
    for pt, pp in zip(prob_true, prob_pred):
        calibration_records.append({
            "class": class_idx,
            "predicted_prob_bin": float(pp),
            "true_prob_bin": float(pt)
        })
calibration_df = pd.DataFrame(calibration_records)
calibration_df.to_csv(f"{model_dir}/calibration_data.csv", index=False)

# ---------------------------
# 5) Optional deeper analysis: reliability plots + robustness + XAI
# ---------------------------
acc_noisy, acc_rotated = None, None
conv_layer = None
shap_values, test_images = None, None

if RUN_ANALYSIS:
    # ---- 5.1 Reliability diagram plots (saved as PNG)
    for class_idx in range(y_pred.shape[1]):
        y_true_bin = (y_true == class_idx).astype(int)
        y_prob = y_pred[:, class_idx]
        prob_true, prob_pred = calibration_curve(y_true_bin, y_prob, n_bins=10)

        plt.figure()
        plt.plot(prob_pred, prob_true, marker="o", label=f"Class {class_idx}")
        plt.plot([0, 1], [0, 1], linestyle="--", color="gray")
        plt.xlabel("Predicted Probability")
        plt.ylabel("True Probability")
        plt.title(f"Reliability Diagram - Class {class_idx}")
        plt.legend()
        plt.savefig(f"plots/reliability_diagram_class_{class_idx}.png", bbox_inches="tight")
        plt.close()

    # ---- 5.2 Robustness: noise + rotation
    def add_noise(image):
        noise = tf.random.normal(shape=tf.shape(image), mean=0.0, stddev=0.1)
        return tf.clip_by_value(image + noise, 0.0, 1.0)

    def rotate_image(image):
        return tf.image.rot90(image, k=1)

    test_ds_noisy = test_ds.map(lambda x, y: (add_noise(x), y))
    test_ds_rotated = test_ds.map(lambda x, y: (rotate_image(x), y))

    acc_noisy = model.evaluate(test_ds_noisy, verbose=0)[1]
    acc_rotated = model.evaluate(test_ds_rotated, verbose=0)[1]
    print(f"Original Acc: {test_metrics.get('accuracy')}, Noisy: {acc_noisy}, Rotated: {acc_rotated}")

    # ---- 5.3 XAI: Grad-CAM + SHAP
    import tf_explain
    import shap

    explainer = tf_explain.core.grad_cam.GradCAM()

    base_model = model.layers[0]


    # Get last conv layer from inner model
    conv_layer = get_last_conv_layer(base_model)
    print(f"[Best selected model: {best_model_name}] Target Conv Layer: {conv_layer.name}")

    # Grad-CAM for first N samples
    N_CAM = min(5, len(test_paths))
    gradcam_meta = []
    for i in range(N_CAM):
        image_path = test_paths[i]
        img_tensor = preprocess_image(image_path, test_labels[i])[0]
        image = np.expand_dims(img_tensor.numpy(), 0)

        # Use your custom Grad-CAM function
        heatmap, _ = make_gradcam_heatmap(base_model, image, conv_layer, class_index=int(test_labels[i]))

        # Save overlay using your function
        save_gradcam_overlay(img_tensor.numpy(), heatmap, f"xai_outputs/gradcam_{i}.png")

        plt.figure()
        plt.imshow(heatmap, cmap="jet")
        plt.axis("off")
        plt.savefig(f"xai_outputs/gradcam_heatmap_{i}.png", bbox_inches="tight")
        plt.close()

        gradcam_meta.append({
            "index": i,
            "image_path": image_path,
            "true_label": int(test_labels[i]),
            "conv_layer_used": conv_layer.name,
            "saved_file": f"xai_outputs/gradcam_{i}.png"
        })


    pd.DataFrame(gradcam_meta).to_csv("xai_outputs/gradcam_metadata.csv", index=False)

    # LIME (faster alternative to SHAP)
    from lime import lime_image
    from skimage.segmentation import mark_boundaries
    explainer_lime = lime_image.LimeImageExplainer()
    lime_results = []
    for i, (p, l) in enumerate(zip(test_paths[:5], test_labels[:5])):
        img = preprocess_image(p, l)[0].numpy()
        explanation = explainer_lime.explain_instance(
            img.astype(np.float64),
            lambda x: model.predict(x, verbose=0),
            top_labels=1, hide_color=0, num_samples=500
        )
        temp, mask = explanation.get_image_and_mask(
            explanation.top_labels[0], positive_only=True, num_features=5, hide_rest=False
        )
        temp_vis = (temp - temp.min()) / (temp.max() - temp.min() + 1e-8)
        plt.figure(); plt.imshow(mark_boundaries(temp_vis, mask)); plt.axis("off")
        plt.savefig(f"xai_outputs/lime_{i}.png", bbox_inches='tight'); plt.close()
        lime_results.append({
            "index": i,
            "image_path": p,
            "true_label": int(l),
            "lime_mask_sum": int(mask.sum()),
            "saved_file": f"xai_outputs/lime_{i}.png"
        })
    import pandas as pd
    pd.DataFrame(lime_results).to_csv("xai_outputs/lime_metadata.csv", index=False)

# 5.4 Save robustness CSV (always, even if RUN_ANALYSIS=False)
robustness_df = pd.DataFrame([{
    "original_accuracy": float(test_metrics.get("accuracy")) if test_metrics.get("accuracy") is not None else None,
    "noisy_accuracy": float(acc_noisy) if acc_noisy is not None else None,
    "rotated_accuracy": float(acc_rotated) if acc_rotated is not None else None
}])
robustness_df.to_csv(f"{model_dir}/robustness_results.csv", index=False)

# ---------------------------
# 6) Aggregate + stats across models/folds (faithfulness + accuracy/f1)
# ---------------------------
# You must populate this elsewhere: {model_name: DataFrame with 'comprehensiveness' column}
faithfulness_results_dict = {}  # <-- fill this if you have it

# 6.1 Faithfulness ANOVA + pairwise independent t-tests + save CSVs
if len(faithfulness_results_dict) >= 2:
    all_comp = []
    for model_name, df in faithfulness_results_dict.items():
        if "comprehensiveness" not in df.columns:
            continue
        for v in df["comprehensiveness"].dropna().tolist():
            all_comp.append({"model": model_name, "comprehensiveness": float(v)})

    all_comp_df = pd.DataFrame(all_comp)
    all_comp_df.to_csv("results/aggregate_faithfulness_metrics.csv", index=False)

    # ANOVA
    anova_result = stats.f_oneway(
        *[df["comprehensiveness"].dropna().values
          for df in faithfulness_results_dict.values()
          if "comprehensiveness" in df.columns]
    )
    pd.DataFrame([{
        "F_statistic": float(anova_result.statistic),
        "p_value": float(anova_result.pvalue)
    }]).to_csv("results/anova_comprehensiveness.csv", index=False)

    # Pairwise independent t-tests
    pairwise_comp = []
    keys = list(faithfulness_results_dict.keys())
    for i in range(len(keys)):
        for j in range(i + 1, len(keys)):
            m1, m2 = keys[i], keys[j]
            if "comprehensiveness" not in faithfulness_results_dict[m1].columns:
                continue
            if "comprehensiveness" not in faithfulness_results_dict[m2].columns:
                continue

            t, p = stats.ttest_ind(
                faithfulness_results_dict[m1]["comprehensiveness"].dropna().values,
                faithfulness_results_dict[m2]["comprehensiveness"].dropna().values,
                equal_var=False
            )
            pairwise_comp.append({
                "model_1": m1,
                "model_2": m2,
                "t_statistic": float(t),
                "p_value": float(p)
            })

    pd.DataFrame(pairwise_comp).to_csv("results/pairwise_ttests_comprehensiveness.csv", index=False)

# 6.2 Summary CSV across CV results dict
summary_df = pd.DataFrame()
for model_name in results:
    fold0 = results[model_name][0]
    mean_metrics = {}
    for k in fold0:
        if k == "confusion_matrix":
            continue
        vals = [r.get(k) for r in results[model_name]]
        vals = [v for v in vals if isinstance(v, (int, float, np.floating, np.integer))]
        if len(vals) > 0:
            mean_metrics[k] = float(np.mean(vals))
    mean_metrics["model"] = model_name
    summary_df = pd.concat([summary_df, pd.DataFrame([mean_metrics])], ignore_index=True)

summary_df.to_csv("results/summary.csv", index=False)
print(summary_df)

# 6.3 Paired t-tests across folds for accuracy and f1 + save CSV
pairwise_records = []
for metric in ["accuracy", "f1"]:
    scores = {}
    for model_name in results:
        vals = [r.get(metric) for r in results[model_name]]
        vals = [v for v in vals if isinstance(v, (int, float, np.floating, np.integer))]
        scores[model_name] = vals

    model_names = list(models_dict.keys())
    for i in range(len(model_names)):
        for j in range(i + 1, len(model_names)):
            m1, m2 = model_names[i], model_names[j]
            if m1 not in scores or m2 not in scores:
                continue
            if len(scores[m1]) == 0 or len(scores[m2]) == 0:
                continue
            if len(scores[m1]) != len(scores[m2]):
                print(f"Skipping paired t-test {m1} vs {m2} for {metric}: fold counts differ ({len(scores[m1])} vs {len(scores[m2])})")
                continue

            stat, p = ttest_rel(scores[m1], scores[m2])
            pairwise_records.append({
                "metric": metric,
                "model_1": m1,
                "model_2": m2,
                "t_statistic": float(stat),
                "p_value": float(p)
            })
            print(f"{m1} vs {m2} {metric}: p={p:.4f}")

pd.DataFrame(pairwise_records).to_csv("results/pairwise_ttests.csv", index=False)

print("All test-related values saved successfully.")
